In [1]:
# import statements
import pandas as pd
import requests
import json

import matplotlib.pyplot as plt


## Data Handling + Split

In [2]:
#call data and clean (remove null and duplicate values)
df_old = pd.read_csv('C:/Users/minju/Desktop/DS3000-26Summer/Farmers-Market/datasets/CropRecc.csv')
df_old = df_old.drop_duplicates()

missing_codes = ["--", "", " ", "nan", "NaN", "None"] #account for different possible null combos


for col in df_old.columns:
    if df_old[col].dtype == object:  # only clean string/object columns
        df_old[col] = df_old[col].str.strip().replace(missing_codes, pd.NA)
    else:
        df_old[col] = df_old[col].replace(missing_codes, pd.NA) 

df_old = df_old.dropna()  # remove rows with any null values

print(df_old.columns)
df_old.head()
print(df_old.dtypes)


Index(['CROPS', 'TYPE_OF_CROP', 'SOIL', 'SEASON', 'SOWN', 'HARVESTED',
       'WATER_SOURCE', 'SOIL_PH', 'SOIL_PH_HIGH', 'CROPDURATION',
       'CROPDURATION_MAX', 'TEMP', 'MAX_TEMP', 'WATERREQUIRED',
       'WATERREQUIRED_MAX', 'RELATIVE_HUMIDITY', 'RELATIVE_HUMIDITY_MAX', 'N',
       'N_MAX', 'P', 'P_MAX', 'K', 'K_MAX'],
      dtype='object')
CROPS                     object
TYPE_OF_CROP              object
SOIL                      object
SEASON                    object
SOWN                      object
HARVESTED                 object
WATER_SOURCE              object
SOIL_PH                  float64
SOIL_PH_HIGH             float64
CROPDURATION             float64
CROPDURATION_MAX           int64
TEMP                     float64
MAX_TEMP                   int64
WATERREQUIRED            float64
WATERREQUIRED_MAX          int64
RELATIVE_HUMIDITY        float64
RELATIVE_HUMIDITY_MAX      int64
N                        float64
N_MAX                      int64
P                        f

In [ ]:
# crop label summary
print("Total number of unique crops:", df_old['CROPS'].nunique())

print("\nCrop types:")
print(df_old['CROPS'].unique().tolist())

print("\nCount of each crop:")
# .to_string() prints every row instead of pandas truncating with "..."
print(df_old['CROPS'].value_counts().to_string())

In [ ]:
# how many unique crops are in each crop category
crops_per_category = (
    df_old.groupby('TYPE_OF_CROP')['CROPS']
    .nunique()
    .sort_values(ascending=False)
)

print("Number of unique crops in each category:")
print(crops_per_category.to_string())

# bar chart
crops_per_category.plot(kind='bar', title='Unique crops per category')
plt.xlabel('Crop category')
plt.ylabel('Number of unique crops')
plt.tight_layout()
plt.show()

In [7]:
maxxArr = df_old[['CROPDURATION_MAX', 'MAX_TEMP',
       'WATERREQUIRED_MAX', 'RELATIVE_HUMIDITY_MAX',
       'N_MAX', 'P_MAX', 'K_MAX', 'SOIL_PH_HIGH']].copy()

print(maxxArr)
maxxArr.head()

       CROPDURATION_MAX  MAX_TEMP  WATERREQUIRED_MAX  RELATIVE_HUMIDITY_MAX  \
0                   150        40               2500                     80   
1                   150        40               2500                     80   
2                   150        40               2500                     80   
3                   150        40               2500                     80   
4                   150        40               2500                     80   
...                 ...       ...                ...                    ...   
56995                90        25                750                     75   
56996                90        25                750                     75   
56997                90        25                750                     75   
56998                90        25                750                     75   
56999                90        25                750                     75   

       N_MAX  P_MAX  K_MAX  SOIL_PH_HIGH  
0       

,CROPDURATION_MAX,MAX_TEMP,WATERREQUIRED_MAX,RELATIVE_HUMIDITY_MAX,N_MAX,P_MAX,K_MAX,SOIL_PH_HIGH
0,150,40,2500,80,100,60,60,8.0
1,150,40,2500,80,100,60,60,8.0
2,150,40,2500,80,100,60,60,8.0
3,150,40,2500,80,100,60,60,8.0
4,150,40,2500,80,100,60,60,8.0


In [3]:
#check correlation 
correlations = df_old.corr(numeric_only=True)
print(correlations)

                        SOIL_PH  SOIL_PH_HIGH  CROPDURATION  CROPDURATION_MAX  \
SOIL_PH                1.000000      0.664134      0.100256          0.100767   
SOIL_PH_HIGH           0.664134      1.000000      0.199739          0.202836   
CROPDURATION           0.100256      0.199739      1.000000          0.977613   
CROPDURATION_MAX       0.100767      0.202836      0.977613          1.000000   
TEMP                   0.219149      0.272520      0.172977          0.140207   
MAX_TEMP               0.220374      0.282283      0.094633          0.077540   
WATERREQUIRED          0.043510      0.036607      0.307509          0.283561   
WATERREQUIRED_MAX      0.046160      0.074244      0.316633          0.297407   
RELATIVE_HUMIDITY     -0.048196     -0.118704     -0.023659          0.006901   
RELATIVE_HUMIDITY_MAX -0.031097     -0.081447     -0.006653          0.019987   
N                     -0.014683     -0.109038      0.270526          0.304813   
N_MAX                 -0.024

In [ ]:
# How strongly is each numeric feature related to the (categorical) targets?
# .corr() only works on numbers, so for the crop targets we use the
# correlation ratio (eta): 0 = no relationship, 1 = feature fully determined
# by which crop/category the row belongs to.
import numpy as np

feats = ['CROPDURATION', 'TEMP', 'WATERREQUIRED', 'RELATIVE_HUMIDITY', 'N', 'P', 'K']

def eta(cat, num):
    """Correlation ratio between categorical column `cat` and numeric column `num`."""
    g = df_old.groupby(cat)[num]
    grand = df_old[num].mean()
    ss_between = (g.count() * (g.mean() - grand) ** 2).sum()
    ss_total = ((df_old[num] - grand) ** 2).sum()
    return np.sqrt(ss_between / ss_total) if ss_total > 0 else 0.0

eta_table = pd.DataFrame({
    'vs_TYPE_OF_CROP': {f: eta('TYPE_OF_CROP', f) for f in feats},
    'vs_CROPS':        {f: eta('CROPS', f)        for f in feats},
}).sort_values('vs_TYPE_OF_CROP', ascending=False)

print("Correlation ratio (eta) of each feature with the crop targets:")
print(eta_table.round(3).to_string())
print(f"\nTYPE_OF_CROP categories: {df_old['TYPE_OF_CROP'].nunique()}  |  unique CROPS: {df_old['CROPS'].nunique()}")
print("Note: eta vs CROPS is naturally high because there are many (57) classes; "
      "use the ranking, not the raw value. vs_TYPE_OF_CROP (10 classes) is the cleaner discriminator.")
eta_table

In [ ]:
#one-hot encoding to turn categorical value into numerical
#create df_sub and df_sub_drop (clean dataset/create subsets)
#df_sub = df.drop(columns='Crop_Health_Label', axis=1)

df_sub = pd.get_dummies(df_old, columns=['TYPE_OF_CROP', 'SOIL', 'SOWN', 'HARVESTED', 'WATER_SOURCE', 'SEASON'])

# drop the target column from features
df_sub = df_sub.drop(columns=['CROPS'])
# Save all one-hot encoded column names.
# NOTE: exclude SOIL_PH / SOIL_PH_HIGH — they are numeric columns that happen
# to start with 'SOIL_', so the prefix match below would wrongly treat them as
# one-hot columns and inflate the feature vectors by 2 (the cause of the
# 71-vs-69 mismatch seen in the API).
ohe_cols = [col for col in df_sub.columns 
            if col.startswith(( 'CROPS_','TYPE_OF_CROP_', 'SOIL_', 'SOWN_', 'HARVESTED_', 'WATER_SOURCE_','SEASON'))
            and col not in ('SOIL_PH', 'SOIL_PH_HIGH')]

ohe_data = df_sub[ohe_cols].copy()  # save the actual data

# Drop from df
df_sub = df_sub.drop(columns=ohe_cols)

#all one value (not helpful for pred)
columnsToDrop = ['CROPDURATION_MAX', 'MAX_TEMP',
       'WATERREQUIRED_MAX', 'RELATIVE_HUMIDITY_MAX',
       'N_MAX','P_MAX', 'K_MAX',
       'SOIL_PH', 'SOIL_PH_HIGH']

df_sub_drop = df_sub.drop(columns=columnsToDrop, axis=1)

#print(df_sub_drop.columns)

df_sub_drop.head()
#df_sub.head()

In [5]:
df_sub_drop.shape

(57000, 7)

In [6]:
from sklearn.model_selection import train_test_split
#split continuous and binary data at same time, consistent rand state
y_vec = df_old['CROPS'].to_numpy()

X_train_cont, X_test_cont, y_train, y_test = train_test_split(
    df_sub_drop.to_numpy(), y_vec, test_size=0.3, random_state=3000
)
binary_train, binary_test, _, _ = train_test_split(
    ohe_data, y_vec, test_size=0.3, random_state=3000
)

In [7]:
import numpy as np
from sklearn.preprocessing import StandardScaler

#scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_cont)
X_test_scaled = scaler.transform(X_test_cont)

#add binary columns back

# df_sub = pd.concat([df_sub, ohe_data], axis=1)

X_train = np.concatenate([X_train_scaled, binary_train.to_numpy().astype(float)], axis=1)
X_test = np.concatenate([X_test_scaled, binary_test.to_numpy().astype(float)], axis=1)

#add column of y-intercept. ONLY FOR LOG REG!
ones = np.ones((X_train.shape[0], 1), dtype=X_train.dtype)
X_train_reg = np.hstack([ones, X_train])

ones = np.ones((X_test.shape[0], 1), dtype=X_test.dtype)
X_test_reg = np.hstack([ones, X_test])

#pd.DataFrame(X_train_scaled).head()
#pd.DataFrame(X_test_scaled).head()

pd.DataFrame(X_test_reg).head()



,0,1,2,3,4,5,6,7,8,9,...,62,63,64,65,66,67,68,69,70,71
0,1.0,-0.920322,-0.455600,0.148607,0.323797,-0.142264,0.276985,-0.190717,7.8,8.5,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
1,1.0,-0.444273,-0.592171,-1.433299,0.461592,-0.695699,-0.932906,-0.800883,7.0,8.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
2,1.0,0.031776,-0.216599,0.310502,0.702734,0.351980,0.125749,-0.553911,6.2,6.7,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,1.0,-0.305033,-0.984815,-0.564345,0.095573,-0.189617,-1.322630,-1.207661,6.1,7.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
4,1.0,-0.528945,-0.080028,0.338840,0.293654,0.547310,0.486389,0.880706,6.3,7.5,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


## KNN Model

In [8]:
from collections import Counter
def knn_cos_with_loop(X_train, y_train, X_test, k):

    y_pred = []
    nearest_neighbors = []

    train_norms = np.linalg.norm(X_train, axis=1, keepdims=True)
    
    X_train_norm = X_train / train_norms
    #i = 0
    for testp in X_test:
        #i += 1
        #print(f'The current iteration is: {i}', end='\r')

        norm_test = testp/np.linalg.norm(testp)
        similarities = norm_test @ X_train_norm.T
        #print(similarities.shape)
            
        neighbor_indices = np.argsort(similarities)[-k:]
        del similarities
        #print(neighbor_indices)
        neighbor_values = y_train[neighbor_indices]
        #print(neighbor_values)

        # Majority vote instead of mean (y_train is categorical)
        most_common = Counter(neighbor_values).most_common(1)[0][0]#among knn, whichever crop label appears most often is the prediction (maj vote)
        y_pred.append(most_common)
        nearest_neighbors.append(neighbor_indices)
    
    return np.array(y_pred), nearest_neighbors

In [9]:
X_train.shape

(39900, 71)

## plot data with ks and model

In [12]:
import subprocess
subprocess.run(["pip", "install", "--upgrade", "nbformat"], check=True)

CompletedProcess(args=['pip', 'install', '--upgrade', 'nbformat'], returncode=0)

In [15]:
#embed plot
from sklearn.metrics import accuracy_score
import plotly.graph_objects as go

k_values = [1,2,3,4,5,6,7,8,9,10]
accuracy_values = []

for k in k_values:
    print(f'We are beginning the modeling of k={k}')
    y_pred, _ = knn_cos_with_loop(X_train, y_train, X_test, k)
    acc = accuracy_score(y_test, y_pred)
    accuracy_values.append(acc)

fig = go.Figure()
fig.add_trace(go.Scatter(x=k_values, y=accuracy_values, mode='lines+markers'))
fig.update_layout(
    title='Accuracy of k-NN Classifier with Cosine Similarity',
    xaxis_title='k (number of neighbors)',
    yaxis_title='Accuracy',
    xaxis=dict(tickmode='array', tickvals=k_values)
)

fig.write_html('accuracy_plot.html', full_html=False, include_plotlyjs='cdn')
fig.show(renderer="browser")

We are beginning the modeling of k=1
We are beginning the modeling of k=2
We are beginning the modeling of k=3
We are beginning the modeling of k=4
We are beginning the modeling of k=5
We are beginning the modeling of k=6
We are beginning the modeling of k=7
We are beginning the modeling of k=8
We are beginning the modeling of k=9
We are beginning the modeling of k=10


In [ ]:
from sklearn.metrics import accuracy_score

#loop through k and print acc
k_values = [1, 2, 3, 4, 5,6,7,8,9,10]
accuracy_values = []

for k in k_values:
    print(f'We are beginning the modeling of k={k}')
    y_pred, _ = knn_cos_with_loop(X_train, y_train, X_test, k)
    acc = accuracy_score(y_test, y_pred)
    accuracy_values.append(acc)

plt.figure(figsize=(10, 6))
plt.plot(k_values, accuracy_values, marker='o')
plt.title('Accuracy of k-NN Classifier with Cosine Similarity')
plt.xlabel('k (number of neighbors)')
plt.ylabel('Accuracy')
plt.xticks(k_values)
plt.grid()
plt.show()

We are beginning the modeling of k=1


KeyboardInterrupt: 

We are beginning the modeling of k=1


NameError: name 'knn_cos_with_loop' is not defined

In [ ]:
# hard coded values
print(df_sub_drop.mean())
print("soil ph",df_old['SOIL_PH'].mean())

print(df_old['SOIL'].value_counts().index[0])  # most frequent soil type

CROPDURATION         109.347488
TEMP                  24.757740
WATERREQUIRED        887.262882
RELATIVE_HUMIDITY     71.197588
N                     81.284277
P                     48.197891
K                     56.605086
dtype: float64
soil ph 6.624366666666667
sandy Loamy soil


In [ ]:
def predict(N, P, K, TYPE_OF_CROP, TEMPERATURE, SEASON, SOWN, HARVESTED, WATER_SOURCE, RELATIVE_HUMIDITY, k=3):
    """
    Returns a single crop prediction given input features.

    Returns:
        str: predicted crop name (CROPS)

    hard-coded values: CROPDURATION, WATERREQUIRED, SOIL, SOIL_PH
    
    Collect: 'TYPE_OF_CROP', 'SEASON', 'SOWN', 'HARVESTED',
       'WATER_SOURCE', 'TEMP', 'RELATIVE_HUMIDITY', 'N', 'P', 'K'
    """
     # hardcoded averages from training data
    CROPDURATION_AVG=109.347
    WATERREQUIRED_AVG = 887.26
    SOIL_PH_AVG = 6.624
    SOIL_AVG = 'sandy Loamy soil'

    x = np.array([
    float(CROPDURATION_AVG),
    float(TEMPERATURE),
    float(WATERREQUIRED_AVG),
    float(RELATIVE_HUMIDITY),
    float(N),
    float(P),
    float(K),
])

    # scale w training scaler
    x_scaled = scaler.transform(x.reshape(1, -1))

    # build OHE vector (all zeros, then flip matching columns to 1)
    ohe_input = np.zeros((1, len(ohe_cols)))

    # map input values to their OHE column names {'SEASON': 'kharif', 'SOWN': 'june', ...}
    categorical_inputs = {
        'TYPE_OF_CROP': TYPE_OF_CROP,
        'SEASON': SEASON,
        'SOWN': SOWN,
        'HARVESTED': HARVESTED,
        'WATER_SOURCE': WATER_SOURCE,
    }
    #loop through each categ. + combines to 'SEASON_kharif'
    for prefix, value in categorical_inputs.items():
        col_name = f'{prefix}_{value}'
        if col_name in ohe_cols: #locates where in binary col and flip to 1 (to encode)
            ohe_input[0, ohe_cols.index(col_name)] = 1
        else:
            raise ValueError(f"Unknown value '{value}' for '{prefix}'. Check OHE column names.")

    # concatenate continuous + binary
    new_X = np.concatenate([x_scaled, ohe_input], axis=1)

    # run knn model
    prediction, _ = knn_cos_with_loop(X_train, y_train, new_X, k)

    return str(prediction[0])

In [ ]:
result = predict(
    N=50,
    P=30,
    K=20,
    TYPE_OF_CROP='cereals',
    TEMPERATURE=25,
    SEASON='kharif',
    SOWN='Jun',
    HARVESTED='Oct',
    WATER_SOURCE='rainfed',
    RELATIVE_HUMIDITY=60
)
print(f'Predicted crop: {result}')

Predicted crop: varagu


In [17]:
print("scaler mean",scaler.mean_.tolist())
print("scaler.scale",scaler.scale_.tolist())
print("ohe",ohe_cols)

scaler mean [109.41124310776961, 24.768781954887128, 889.3354862155351, 71.18053132832067, 81.30695488721855, 48.238172932330734, 56.651092731829635]
scaler.scale [53.14582187119277, 5.857734417104357, 356.40542952108115, 23.22281665629778, 33.78898195795276, 17.191630190056443, 27.533471870470983]
ohe ['SOIL_PH', 'SOIL_PH_HIGH', 'TYPE_OF_CROP_Root&tuber', 'TYPE_OF_CROP_bulbvegetables', 'TYPE_OF_CROP_cereals', 'TYPE_OF_CROP_colecrops', 'TYPE_OF_CROP_fibre crop', 'TYPE_OF_CROP_millets', 'TYPE_OF_CROP_oil seeds', 'TYPE_OF_CROP_pulses', 'TYPE_OF_CROP_sugar crops', 'TYPE_OF_CROP_vegetables', 'SOIL_Alluvial soil', 'SOIL_Black Soil', 'SOIL_Clay soil', 'SOIL_Laterite soil', 'SOIL_Loamy soil', 'SOIL_Red soil', 'SOIL_Sandy soil', 'SOIL_Sandy\xa0soil', 'SOIL_black cotton soil', 'SOIL_brown Loamy soil', 'SOIL_clay Loamy soil', 'SOIL_cotton\xa0soil', 'SOIL_deep soil', 'SOIL_friable soil', 'SOIL_heavy Black Soil', 'SOIL_heavy soil', 'SOIL_light Loamy soil', 'SOIL_light soi', 'SOIL_loamy\xa0soil', '

In [18]:
import json

rows = []
for i in range(len(X_train)):
    vec = X_train[i].tolist()
    label = y_train[i]
    rows.append(f"('{json.dumps(vec)}', '{label}')")

sql = "INSERT INTO model3_training_data (feature_vector, crop_label) VALUES\n"
sql += ",\n".join(rows) + ";"

with open("model3_training_inserts.sql", "w") as f:
    f.write(sql)

print("done — check model3_training_inserts.sql")

done — check model3_training_inserts.sql
